# 01 — Nettoyage des emails (v2)

Objectif : préparer `phishing_email.csv` avant le split train/test.

**Ce qui change par rapport à la v1** (deux ajouts, qui rendent l'évaluation plus honnête) :
- on détecte les textes qui apparaissent **avec les deux étiquettes** (phishing *et* légitime) : leur label est contradictoire, on les retire au lieu de garder « le premier » au hasard ;
- on calcule un identifiant de **groupe de quasi-doublons** (`dup_group`) : des emails presque identiques (réponses citées, transferts, spam à gabarit) ne doivent jamais se retrouver à la fois en train et en test. Le split du notebook 02 s'appuiera sur ce groupe.

- on retire les **newsletters CNN** (≈ 4 500 emails au gabarit identique, étiquetés phishing à ≈ 99 %) : ce sont des alertes d'actualité, pas du phishing ; les garder apprendrait au modèle « pied de page CNN = phishing ».

Étapes :
1. Charger et isoler les colonnes utiles
2. Valeurs manquantes / textes vides
3. Retirer les newsletters CNN mal étiquetées
4. Doublons exacts et étiquettes contradictoires
5. Équilibre des classes
6. Groupes de quasi-doublons
7. Sauvegarder la version propre

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)

## 1. Charger les données et isoler les colonnes utiles

Le fichier brut ne contient que deux colonnes utiles : `text_combined` (sujet + corps fusionnés) et `label` (1 = phishing, 0 = légitime).

In [2]:
df = pd.read_csv("../data/raw/phishing_email.csv")
print("Shape:", df.shape)
df.head(3)

Shape: (82486, 2)


,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls,0
1,nom actual vols 24 th forwarded sabrae zajac hou ect 05 30 2001 12 07 pm enr...,0
2,enron actuals march 30 april 1 201 estimated actuals march 30 2001 flow marc...,0


## 2. Valeurs manquantes et textes vides

Une valeur manquante ou un texte vide dans `text_combined` casserait le feature engineering (division par `len(body)` nul, par exemple) et n'apporte aucun signal.

In [3]:
print(df.isna().sum())
print()
empty_mask = df["text_combined"].fillna("").str.strip() == ""
print("Lignes avec texte vide ou NaN:", empty_mask.sum())

text_combined    0
label            0
dtype: int64

Lignes avec texte vide ou NaN: 1


In [4]:
df = df[~empty_mask].reset_index(drop=True)
print("Shape après suppression des textes vides:", df.shape)

Shape après suppression des textes vides: (82485, 2)


## 3. Retirer les newsletters CNN mal étiquetées

L'analyse des tokens quasi purs (notebook 02, section 5b) a montré un bloc d'environ 4 500 emails contenant `cnncom`, `manage settings`, `unsubscribe`, `privacy guidelines`, `time warner`… : le pied de page des alertes d'actualité de CNN. Ces emails sont étiquetés **phishing à ≈ 99 %** (4 472 sur 4 526 sur ce corpus). Une alerte d'actualité envoyée à un abonné n'est pas du phishing : soit l'étiquette est fausse, soit c'est un artefact de collecte du corpus. Dans les deux cas, les garder apprend au modèle un raccourci (« gabarit CNN = phishing ») qui ne sert à rien sur de vrais emails, et fausse le score.

On les repère par la présence de la chaîne `cnncom` (qui attrape aussi les URLs collées comme `httpwwwcnncomprivacyhtml`). Le filtre est volontairement simple : il emporte aussi une cinquantaine d'emails étiquetés légitimes (alertes « cnn alerts » au même gabarit, messages de listes de diffusion qui citent une URL cnn.com). C'est un coût marginal (≈ 1 % du bloc retiré), accepté pour ne pas laisser de raccourci de gabarit ; il reste en outre ≈ 350 emails qui citent « cnn » sans `cnncom`, traités au notebook 03 via `SOURCE_MARKERS`. **Vérifie les exemples affichés** avant de valider : si tu n'es pas d'accord, passe `REMOVE_CNN_NEWSLETTERS` à `False`.

In [5]:
REMOVE_CNN_NEWSLETTERS = True

cnn_mask = df["text_combined"].str.contains("cnncom", regex=False)
print(f"Emails contenant le gabarit CNN ('cnncom') : {cnn_mask.sum()}")
print("Étiquettes dans ce groupe :")
print(df.loc[cnn_mask, "label"].value_counts().to_string())
if cnn_mask.any():
    print(f"Part étiquetée phishing : {df.loc[cnn_mask, 'label'].mean():.1%}\n")
    for t in df.loc[cnn_mask, "text_combined"].sample(min(3, int(cnn_mask.sum())), random_state=0):
        print("-", t[:200])

Emails contenant le gabarit CNN ('cnncom') : 4526
Étiquettes dans ce groupe :
label
1    4472
0      54
Part étiquetée phishing : 98.8%

- daily top 10 rkwijndepicpakcom daily top 10 cnncom top videos stories aug 1 2008 358 pm edt top 10 videos 1 paris hilton takes mccain httpwwwcnncomvideopartnersemailindexhtmlurlvideopolitics20080806wy
- daily top 10 inthavongajekkatguymonokorg daily top 10 cnncom top videos stories aug 1 2008 358 pm edt top 10 videos 1 mom parties club httpwwwcnncomvideopartnersemailindexhtmlurlvideocrime20080805ngmo
- daily top 10 punkies_1976gbcaustraliacomau daily top 10 cnncom top videos stories aug 1 2008 358 pm edt top 10 videos 1 montauk monster httpwwwcnncomvideopartnersemailindexhtmlurlvideous20080731moosmo


In [6]:
if REMOVE_CNN_NEWSLETTERS:
    df = df[~cnn_mask].reset_index(drop=True)
    print(f"\nShape après retrait des newsletters CNN : {df.shape}")
else:
    print("Newsletters CNN conservées (REMOVE_CNN_NEWSLETTERS = False).")


Shape après retrait des newsletters CNN : (77959, 2)


## 4. Doublons exacts et étiquettes contradictoires

Deux problèmes distincts :

- **Doublon exact, même label** : le même email apparaît plusieurs fois. S'il se retrouve en train *et* en test, le modèle « triche » en mémorisant. On garde une seule copie.
- **Même texte, labels différents** : on ne sait pas lequel est juste. `drop_duplicates(keep="first")` garderait l'un des deux au hasard de l'ordre du fichier. On retire **toutes** les copies de ces textes ambigus.

On traite les contradictions d'abord, puis les doublons.

In [7]:
n_start = len(df)

labels_per_text = df.groupby("text_combined")["label"].nunique()
conflict_texts = labels_per_text[labels_per_text > 1].index
print(f"Textes présents avec les DEUX étiquettes : {len(conflict_texts)}")

if len(conflict_texts) > 0:
    ex = df[df["text_combined"].isin(conflict_texts)].sort_values("text_combined").head(4)
    display_cols = ex.assign(text_combined=ex["text_combined"].str[:70])
    print(display_cols.to_string())

n_conflict_rows = int(df["text_combined"].isin(conflict_texts).sum())
df = df[~df["text_combined"].isin(conflict_texts)].reset_index(drop=True)
print(f"Lignes retirées pour étiquette contradictoire : {n_conflict_rows}")

Textes présents avec les DEUX étiquettes : 0
Lignes retirées pour étiquette contradictoire : 0


In [8]:
dup_mask = df.duplicated(subset=["text_combined"], keep="first")
print(f"Doublons exacts (copies en trop) : {dup_mask.sum()}")
print("Répartition des copies retirées par classe :")
print(df.loc[dup_mask, "label"].value_counts().to_string())

df = df.loc[~dup_mask].reset_index(drop=True)
print(f"\nAvant : {n_start} lignes -> après contradictions + doublons : {len(df)} lignes")

Doublons exacts (copies en trop) : 408
Répartition des copies retirées par classe :
label
0    362
1     46

Avant : 77959 lignes -> après contradictions + doublons : 77551 lignes


## 5. Rechecker l'équilibre des classes

Le dédoublonnage peut changer le ratio phishing/légitime si les doublons ne sont pas répartis également entre les deux classes.

In [9]:
print(df["label"].value_counts().to_string())
print()
print(df["label"].value_counts(normalize=True).round(3).to_string())

label
0    39179
1    38372

label
0    0.505
1    0.495


## 6. Groupes de quasi-doublons

Les doublons *exacts* ne sont que la partie visible. Dans ce corpus, beaucoup d'emails sont presque identiques : réponses qui citent le message précédent, transferts, spam envoyé en masse avec un gabarit. Si une copie est en train et une quasi-copie en test, le score de test est gonflé sans que le modèle ait appris à généraliser.

Solution simple et reproductible : on regroupe les emails dont les **20 premiers mots** sont identiques (le sujet est fusionné en tête de `text_combined`, donc le début est très discriminant pour repérer un même fil ou un même gabarit). Le split (notebook 02) et la validation croisée (notebook 03) garderont chaque groupe d'un seul côté.

Limite assumée : cette règle rate les quasi-doublons dont le début diffère. C'est une amélioration, pas une garantie.

In [10]:
N_WORDS_PREFIX = 20

prefix = df["text_combined"].str.split().str[:N_WORDS_PREFIX].str.join(" ")
df["dup_group"] = pd.factorize(prefix)[0]

group_stats = df.groupby("dup_group")["label"].agg(size="size", phishing_share="mean")
multi = group_stats[group_stats["size"] > 1]

print(f"Nombre de groupes            : {len(group_stats)}")
print(f"Groupes de plus d'un email   : {len(multi)}")
print(f"Emails dans un groupe > 1    : {int(multi['size'].sum())} "
      f"({multi['size'].sum() / len(df):.1%} du corpus)")
print(f"Taille du plus gros groupe   : {int(group_stats['size'].max())}")
mixed = ((multi["phishing_share"] > 0) & (multi["phishing_share"] < 1)).sum()
print(f"Groupes mélangeant les 2 classes : {mixed}")

Nombre de groupes            : 74786
Groupes de plus d'un email   : 1502
Emails dans un groupe > 1    : 4267 (5.5% du corpus)
Taille du plus gros groupe   : 65
Groupes mélangeant les 2 classes : 0


In [11]:
largest = group_stats.sort_values("size", ascending=False).head(5)
for gid, row in largest.iterrows():
    example = df.loc[df["dup_group"] == gid, "text_combined"].iloc[0][:90]
    print(f"groupe {gid:>6} | taille {int(row['size']):>5} | part phishing {row['phishing_share']:.2f} | {example}")

groupe  31264 | taille    65 | part phishing 0.00 | ysxshweetyrxxliyysxsliststuxonicenet send tuxoniceusers mailing list submissions enpsfhxzs
groupe  31106 | taille    62 | part phishing 0.00 | ohwwlejspbfotbplistscmswaikatoacnz send wekalist mailing list submissions iuxxdkqklistscms
groupe  31487 | taille    49 | part phishing 0.00 | qvcimnfiqkwxudxmipassocorg send ietfdkim mailing list submissions gfkyhddlmipassocorg subs
groupe   8801 | taille    38 | part phishing 1.00 | graphics software available cheap oem versions good morning offer latest oem packages grap
groupe  33800 | taille    36 | part phishing 1.00 | google adwordsnoreply adwordsnoreplygooglecom dear advertiser unable process payment ads s


## 7. Sauvegarder la version propre

Cette version (dédupliquée, sans contradictions, avec `dup_group`) sert de point de départ au notebook 02 : split groupé + EDA + features.

In [12]:
out_path = Path("../data/processed/emails_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print("Sauvegardé :", df.shape, "->", out_path)

Sauvegardé : (77551, 3) -> ..\data\processed\emails_clean.csv
